# Day 7: MCP Servers (Google Stitch)

This notebook introduces MCP Servers using Google Stitch and presents an end-to-end demo plan for building a website UI and replicating it with an IDE agent.

Goals:
1. Understand MCP Server concept and workflow.
2. Learn basics of Stitch server setup and architecture.
3. Build a minimal UI prototype w/ Google Stitch.
4. Craft step-by-step prompts for Stitch and IDE agent flows.

## 1) Background: MCP Server + Google Stitch + Antigravity

MCP (Model Context Protocol) Servers are a contract-based gateway for LLM agents to call structured tools and manage state. A Stitch MCP Server is a specialized implementation for Google Cloud/Vertex AI that exposes a manifest plus action endpoints to integrate LLMs with app behavior.

Antigravity is a high-level connector library that simplifies Stitch MCP integration by offering unified client APIs and safe request/response handling. In this flow we use a pseudo-SDK (`stitch_antigravity`) for concept planning.

Key concepts:
- **MCP contract**: includes `manifest.json` describing tools, schemas, and supported actions.
- **Stitch manifest endpoint**: `/mcp/manifest` returns tool definitions to the agent.
- **Action endpoint**: `/mcp/action` receives tool calls and returns a deterministic tool output JSON payload.
- **Tool schema**: JSON Schema for validating inputs before running logic.
- **IDE agent**: local IDE-based agent can inspect manifest and generate/maintain implementation using prompts.
- **Codex/Antigravity**: use code intelligence docs to map `openai/assistant` tool calls into Stitch return values.

## 2) Architecture diagram (conceptual)

- Browser/UI -> Stitch MCP Server -> LLM Agent (GPT/other)
- Stitch provides HTTP endpoints for `openai/agent` style tool calls.
- IDE agent can run code generation and local scaffolding from the same tool descriptions.

## 3) High-level awareness: what you can do with Stitch MCP Server

- *Register UI elements as tools* (buttons, forms, meta state).
- *Route interactions* to `ask` calls and structured states.
- *Build website-like flows* with one endpoint to evaluate and generate HTML/CSS.
- *Link agent logic* to frontend actions via tool call results.

## 4) Practical demo plan: UI for website with Google Stitch

1. Create an MCP server manifest and endpoint.
2. Define tools: `generate_page`, `update_component`, `run_in_browser`.
3. Setup a local web server (e.g., Flask/Streamlit) that exposes input + rendered output.
4. Stitch server does state + responses; update UI on every step.
5. Validate with sample flow: *Landing page => About section => Contact form*.

In [ ]:
# Sample structure with Stitch + Antigravity style integration
# (Example uses pseudo-API names; adapt to real SDK details)
from pathlib import Path
import json

manifest = {
    "mcp_version": "1.0",
    "id": "com.aceturtle.stitch",
    "name": "Ace Turtle Stitch MCP",
    "release": "v0.1",
    "tools": [
        {
            "id": "generate_page",
            "name": "Generate website page",
            "description": "Create HTML + CSS for a simple website section",
            "input_schema": {
                "type": "object",
                "properties": {
                    "page_title": {"type": "string"},
                    "sections": {"type": "array", "items": {"type": "string"}},
                    "theme": {"type": "string", "enum": ["light", "dark"]}
                },
                "required": ["page_title", "sections"]
            },
            "output_schema": {
                "type": "object",
                "properties": {
                    "html": {"type": "string"},
                    "css": {"type": "string"},
                    "status": {"type": "string"}
                },
                "required": ["html", "css", "status"]
            }
        }
    ]
}
Path('stitch_manifest.json').write_text(json.dumps(manifest, indent=2))
print('Created manifest for Knit Stitch MCP (manifest saved as stitch_manifest.json)')

# Example Antigravity client usage (pseudo-code):
try:
    from stitch_antigravity import StitchClient
except ImportError:
    print('Install stitch_antigravity with pip if available or replace with your library')

client = None
if 'StitchClient' in globals():
    client = StitchClient(api_key='YOUR_API_KEY', base_url='http://localhost:8000')
    client.register_manifest_from_dict(manifest)
    print('Manifest registered with Antigravity Stitch client')
else:
    print('Manual manifest flow: run local MCP server and expose /mcp/manifest and /mcp/action endpoints')

## 5) Basic usage - step-by-step 

### 5.1 Setup environment
- Python 3.11+ (or locally available).
- `pip install flask requests` (or your preferred local server).
- Create project tree: `day7/`, `day7/mcp_server.py`, `day7/templates/`.

### 5.2 Stitch manifest + endpoint (concept)
- `POST /api/v1/mcp/manifest` serves JSON metadata (tool types and input schema).
- `POST /api/v1/mcp/action` receives requests from Stitch agent and returns structured tool outputs.

### 5.3 UI workflow example
- User enters `Build homepage` request.
- Agent calls `generate_page` tool with page_title+sections.
- Server returns HTML payload.
- Frontend updates with returned code live.

## 6) Stitch prompt guide (for end users)

Prompt 1: Generate manifest skeleton
```
You are a Google Stitch MCP server designer. Generate a JSON manifest with a tool named "generate_page" that accepts page title, sections, and theme. Include `mcp_version`, `id`, and `release` fields.
```

Prompt 2: Build content flow
```
- Create an endpoint `/mcp/action` that interprets tool calls.
- For `generate_page`, return an object with `html` and `css` in `output`.
- Keep all responses JSON-serializable and include `tool_result`.
```

Prompt 3: Run a test scenario
```
Input: page_title = "AI Training Lab" sections = ["Hero", "Features", "Contact"]
Desired: 1) HTML body with sections, 2) inline lightweight CSS, 3) call details.
```

## 7) IDE agent prompt guide (for developer scaffolding)

Prompt 1: Generate scaffold
```
You are an IDE agent. Generate a full Python project under `day7/` that includes:
* `mcp_server.py` (Flask app + endpoints for Stitch manifest/action).
* `README.md` with instructions to run `python mcp_server.py`.
* `frontend.html` template with JavaScript to call `/mcp/action` and render HTML/CSS output.
```

Prompt 2: Implement generate_page action
```
In `mcp_server.py`, implement function `generate_page(payload)` that returns: `{'html': ..., 'css': ..., 'status': 'ok'}`. Use provided `page_title`, `sections`.
```

Prompt 3: Validate and test
```
Run the Flask server locally. Use cURL or browser to POST to `/mcp/action` with input for a simple homepage and verify the returned JSON contains `html`.
```

## 8) Example IDE agent internal flow

- Agent reads `stitch_manifest.json`.
- Agent syncs tooling definitions with local editor file templates.
- Agent creates code (Flask endpoints + UI script).
- Developer checks and refines.

### Pattern: one command per file output
Use the tool schema to generate: `ui.py`, `mcp_config.yaml`, `example_input.json`, and `tests/test_mcp_server.py`.

## 9) Quick reference checklist
- [ ] Manifest has `mcp_version` and `tools` keys
- [ ] Action endpoint includes `tool_id`, `input`, `user_id`
- [ ] Output from tool obeys JSON format and includes `message`/`payload`
- [ ] UI shows immediate HTML/CSS render and logs tool calls
- [ ] Using IDE agent prompts, update files in repo and run fast local test

## 10) Next steps (learning path)
- Deploy Stitch MCP server on cloud (Google Cloud Run, Vercel functions).
- Connect real LLM (OpenAI / Vertex AI) to route through the MCP server.
- Add permissions/auth in manifest for protected workflow.
- Add telemetry events and error handling on client side.